> ### **Sorting Algorithms**

> *Created by Jason Naicker*.

> *[Github](https://github.com/JasonNaicker)*

---

In [51]:
import sys, os, gc
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from DataStructures.Heap import Heap

from typing import Final, Optional, override, Final
from abc import abstractmethod, ABC
from time import perf_counter
from functools import wraps
import random

In [52]:
class SortingTemplate(ABC):
    def __init__(self, items : list[int]) -> None:
        self._original_items = items[:]
        self._items = self._original_items[:]
        self._size = len(self._items)

    @abstractmethod
    def sort(self) -> any:
        '''Implement sorting algorithm'''
        pass

    def print_contents(self) -> None:
        print(f"The Array is now: {self._items} using sorting algorithm {self.__class__.__name__}")

    def prepare_items(self, randomize : bool = False) -> None:
        self._items = self._original_items[:]
        if randomize:
            random.shuffle(self._items)

In [53]:
ITERATIONS : Final[int] = 100
Benchmarks = {}

def myTimer(iterations : int = ITERATIONS):
    def decorator(func): 
        @wraps(func)
        def wrapper(*args, **kwargs):
            times = []
            result = None
            warmup_iterations = iterations // 2

            #Cache warmup
            for _ in range(warmup_iterations):
                func(*args, **kwargs) 

            #Reflection reference
            self_obj = args[0] if args else None
            class_name = self_obj.__class__.__name__ if self_obj else "N/A"

            #Disable GC
            gc_was_enabled = gc.isenabled()
            gc.disable()

            try:
                #Start benchmark
                start = perf_counter()
                for _ in range(iterations):
                    if self_obj:
                        getattr(self_obj, "prepare_items")()
                        
                    result = func(*args, **kwargs)
                end = perf_counter()
                times.append(end - start)

                #Display output
                avg : float = sum(times) / len(times)
                Benchmarks[class_name] = avg
                print(f"{class_name} took {avg:.8f} seconds for {iterations} iterations.")
            except Exception as e:
                raise ValueError(f"Failed to benchmark for {class_name}: {str(e)}")
            finally:
                if gc_was_enabled:
                    gc.enable()

            return result
        return wrapper
    return decorator

> #### **Algos**

In [54]:
class SelectionSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        for i in range(self._size):
            min_index = i
            for j in range(i + 1, self._size):
                if self._items[j] < self._items[min_index]:
                    min_index = j
            self._items[i], self._items[min_index] = self._items[min_index], self._items[i]
        return

In [55]:
class BubbleSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        for i in range(0, self._size - 1):
            for j in range(0, self._size - i - 1):
                if self._items[j] > self._items[j + 1]:
                    self._items[j + 1], self._items[j] = self._items[j], self._items[j + 1]
        return 

In [56]:
class HoareQuickSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

In [57]:
class LomutoQuickSort(SortingTemplate):    
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        self.quick_sort(0, len(self._items) - 1)

    def quick_sort(self, start : int, end : int) -> None:
        if start >= end:
            return

        p = self.partition(start, end)
        self.quick_sort(start, p - 1)
        self.quick_sort(p + 1, end)

    def partition(self, start : int, end : int) -> int:
        pivot = self._items[end]
        i = start - 1

        for j in range(start, end):
            if self._items[j] <= pivot:
                i += 1
                self._items[i], self._items[j] = self._items[j], self._items[i]

        self._items[i + 1], self._items[end] = self._items[end], self._items[i + 1]

        return i + 1

In [58]:
class MergeSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

    def merge_sort(self, start : int, end : int) -> None:
        mid : int = start + (end - start) // 2
        self.merge_sort(start, mid)
        self.merge_sort(mid + 1, end)
        self.merge(start, mid, end )

    def merge(self, start : int, mid : int, end : int) -> None:
        pass

In [59]:
class InsertionSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
       for i in range(1, len(self._items)):
            key = self._items[i]
            j = i - 1
            while j >= 0 and key < self._items[j]:
                self._items[j + 1], self._items[j] = self._items[j], self._items[j+1]
                j -= 1

In [15]:
class ShellSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

In [62]:
class TimSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

In [60]:
class HeapSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        _heap = Heap(self._items, isMinHeap=False)

        arr = _heap.items()
        for i in range(len(arr) - 1, 0, -1):
            _heap._swap(0, i)
            _heap.heapify_down(0, i)

        self._items[:] = _heap._arr

In [61]:
class BucketSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

In [16]:
class CountingSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        min_val : int = min(self._items)
        max_val : int = max(self._items)
        prefix_array : list[int] = [0] * (max_val - min_val + 1)
        result_array : list[int] = [0] * len(self._items)

        #Frequency array
        for value in self._items:
            prefix_array[value - min_val] += 1

        #Prefix array
        for i in range(1, len(prefix_array)):
            prefix_array[i] += prefix_array[i - 1]

        for i in range(len(self._items) - 1, -1, -1):
            index = self._items[i] - min_val
            position = prefix_array[index] - 1
            result_array[position] = self._items[i]
            prefix_array[index] -= 1

        self._items = result_array

In [63]:
class RadixSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

In [64]:
class IntroSort(SortingTemplate):
    @myTimer(ITERATIONS)
    @override
    def sort(self):
        pass

---
> ### **Implementation**

In [24]:
if __name__ == "__main__":
    ARRAY_SIZE : Final[int] = 1000
    
    arr = [random.randint(1,100) for i in range(ARRAY_SIZE)]
    a = CountingSort(arr)
    a.sort()
    a.print_contents()

CountingSort took 0.02038410 seconds for 100 iterations.
The Array is now: [1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 18, 18, 18, 19, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 20, 20, 20, 21, 21, 21, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 23, 23, 23, 23, 23, 23, 23, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 26, 26, 26, 26, 26,